In [0]:
# 02 - Silver Transformation

PROJECT_CATALOG = "demoworkspacejoby"

BRONZE_SCHEMA = "fraud_bronze"

SILVER_SCHEMA = "fraud_silver"

BRONZE_DB = f"{PROJECT_CATALOG}.{BRONZE_SCHEMA}"

SILVER_DB = f"{PROJECT_CATALOG}.{SILVER_SCHEMA}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_DB}")

DataFrame[]

In [0]:
cards_raw_df = spark.table(f"{BRONZE_DB}.cards_raw")
transactions_raw_df = spark.table(f"{BRONZE_DB}.transactions_raw")
users_raw_df = spark.table(f"{BRONZE_DB}.users_raw")
mcc_codes_raw_df = spark.table(f"{BRONZE_DB}.mcc_codes_raw")
fraud_labels_raw_df = spark.table(f"{BRONZE_DB}.fraud_labels_raw")

In [0]:
from pyspark.sql.functions import (
    col, trim, regexp_replace, to_timestamp, to_date,
    when, upper, lower, year, month, dayofweek,
    hour, date_format, lit, coalesce
)

from pyspark.sql.types import (
    IntegerType, DoubleType, BooleanType, LongType, StringType
)

In [0]:
silver_cards_df = (
    cards_raw_df
    .withColumn("id", col("id").cast(IntegerType()))
    .withColumn("client_id", col("client_id").cast(IntegerType()))
    .withColumn("card_brand", trim(col("card_brand")))
    .withColumn("card_type", trim(col("card_type")))
    .withColumn("card_number", trim(col("card_number")))
    .withColumn("expires", trim(col("expires")))
    .withColumn("cvv", col("cvv").cast(IntegerType()))
    .withColumn(
        "has_chip",
        when(upper(trim(col("has_chip"))) == "YES", True)
        .when(upper(trim(col("has_chip"))) == "NO", False)
        .otherwise(None)
        .cast(BooleanType())
    )
    .withColumn("num_cards_issued", col("num_cards_issued").cast(IntegerType()))
    .withColumn("credit_limit", regexp_replace(col("credit_limit"), "[$,]", "").cast(DoubleType()))
    .withColumn("acct_open_date", to_date(col("acct_open_date"), "MM/yyyy"))
    .withColumn("year_pin_last_changed", col("year_pin_last_changed").cast(IntegerType()))
    .withColumn(
        "card_on_dark_web",
        when(lower(trim(col("card_on_dark_web"))) == "yes", True)
        .when(lower(trim(col("card_on_dark_web"))) == "no", False)
        .otherwise(None)
        .cast(BooleanType())
    )
)

silver_cards_df.write.mode("overwrite").saveAsTable(f"{SILVER_DB}.cards")

display(spark.table(f"{SILVER_DB}.cards").limit(5))

id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
4524,825,Visa,Debit,4344676511950444,12/2022,623,true,2,24295.0,2002-09-01,2008,false
2731,825,Visa,Debit,4956965974959986,12/2020,393,true,2,21968.0,2014-04-01,2014,false
3701,825,Visa,Debit,4582313478255491,02/2024,719,true,2,46414.0,2003-07-01,2004,false
42,825,Visa,Credit,4879494103069057,08/2024,693,false,1,12400.0,2003-01-01,2012,false
4659,825,Mastercard,Debit (Prepaid),5722874738736011,03/2009,75,true,1,28.0,2008-09-01,2009,false


In [0]:
silver_users_df = (
    users_raw_df
    .withColumn("id", col("id").cast(IntegerType()))
    .withColumn("current_age", col("current_age").cast(IntegerType()))
    .withColumn("retirement_age", col("retirement_age").cast(IntegerType()))
    .withColumn("birth_year", col("birth_year").cast(IntegerType()))
    .withColumn("birth_month", col("birth_month").cast(IntegerType()))
    .withColumn("gender", trim(col("gender")))
    .withColumn("address", trim(col("address")))
    .withColumn("latitude", col("latitude").cast(DoubleType()))
    .withColumn("longitude", col("longitude").cast(DoubleType()))
    .withColumn("per_capita_income", regexp_replace(col("per_capita_income"), "[$,]", "").cast(DoubleType()))
    .withColumn("yearly_income", regexp_replace(col("yearly_income"), "[$,]", "").cast(DoubleType()))
    .withColumn("total_debt", regexp_replace(col("total_debt"), "[$,]", "").cast(DoubleType()))
    .withColumn("credit_score", col("credit_score").cast(IntegerType()))
    .withColumn("num_credit_cards", col("num_credit_cards").cast(IntegerType()))
)

silver_users_df.write.mode("overwrite").saveAsTable(f"{SILVER_DB}.users")

display(spark.table(f"{SILVER_DB}.users").limit(5))

id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
825,53,66,1966,11,Female,462 Rose Lane,34.150001525878906,-117.76000213623047,29278.0,59696.0,127613.0,787,5
1746,53,68,1966,12,Female,3606 Federal Boulevard,40.7599983215332,-73.73999786376953,37891.0,77254.0,191349.0,701,5
1718,81,67,1938,11,Female,766 Third Drive,34.02000045776367,-117.88999938964844,22681.0,33483.0,196.0,698,5
708,63,63,1957,1,Female,3 Madison Street,40.709999084472656,-73.98999786376953,163145.0,249925.0,202328.0,722,4
1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.7599983215332,-122.44000244140625,53797.0,109687.0,183855.0,675,1


In [0]:
transactions_clean_df = (
    transactions_raw_df
    .withColumn("id", col("id").cast(LongType()))
    .withColumn("transaction_date", to_timestamp(col("date"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("client_id", col("client_id").cast(IntegerType()))
    .withColumn("card_id", col("card_id").cast(IntegerType()))
    .withColumn("amount", regexp_replace(col("amount"), "[$,]", "").cast(DoubleType()))
    .withColumn("use_chip", trim(col("use_chip")))
    .withColumn("merchant_id", col("merchant_id").cast(LongType()))
    .withColumn("merchant_city", trim(col("merchant_city")))
    .withColumn("merchant_state", trim(col("merchant_state")))
    .withColumn("zip", trim(col("zip")))
    .withColumn("mcc", trim(col("mcc").cast(StringType())))
    .withColumn("errors", trim(col("errors")))
    .drop("date")
)

transactions_with_mcc_df = (
    transactions_clean_df
    .join(
        mcc_codes_raw_df.select(
            trim(col("mcc").cast(StringType())).alias("mcc"),
            col("mcc_description")
        ),
        on="mcc",
        how="left"
    )
)

silver_transactions_df = (
    transactions_with_mcc_df
    .join(
        fraud_labels_raw_df.select(
            col("transaction_id"),
            col("is_fraud")
        ),
        transactions_with_mcc_df["id"] == fraud_labels_raw_df["transaction_id"],
        how="left"
    )
    .drop("transaction_id")
    .withColumn("is_fraud", coalesce(col("is_fraud"), lit(0)).cast(IntegerType()))
    .withColumn("transaction_year", year(col("transaction_date")))
    .withColumn("transaction_month", month(col("transaction_date")))
    .withColumn("day_of_week", dayofweek(col("transaction_date")))
    .withColumn("day_name", date_format(col("transaction_date"), "EEEE"))
    .withColumn("hour_of_day", hour(col("transaction_date")))
    .withColumn(
        "time_of_day",
        when((col("hour_of_day") >= 6) & (col("hour_of_day") < 12), "Morning")
        .when((col("hour_of_day") >= 12) & (col("hour_of_day") < 18), "Afternoon")
        .when((col("hour_of_day") >= 18) & (col("hour_of_day") < 22), "Evening")
        .otherwise("Night")
    )
)

silver_transactions_df.write.mode("overwrite").saveAsTable(f"{SILVER_DB}.transactions")

display(spark.table(f"{SILVER_DB}.transactions").limit(5))

mcc,id,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,errors,transaction_date,mcc_description,is_fraud,transaction_year,transaction_month,day_of_week,day_name,hour_of_day,time_of_day
5541,7475350,114,3398,64.0,Swipe Transaction,61195,North Hollywood,CA,91606.0,,2010-01-01T00:38:00.000Z,Service Stations,0,2010,1,6,Friday,0,Night
5411,7475453,1313,4183,2.77,Swipe Transaction,61746,Sutersville,PA,15083.0,,2010-01-01T02:57:00.000Z,"Grocery Stores, Supermarkets",0,2010,1,6,Friday,2,Night
3780,7475464,957,4532,14.21,Swipe Transaction,44795,Marysville,OH,43040.0,,2010-01-01T03:13:00.000Z,Computer Network Services,0,2010,1,6,Friday,3,Night
7538,7475498,1736,113,55.79,Swipe Transaction,24823,Brooklyn,NY,11211.0,,2010-01-01T04:27:00.000Z,Automotive Service Shops,0,2010,1,6,Friday,4,Night
5912,7475528,1541,228,4.22,Swipe Transaction,61806,Clinton,TN,37716.0,,2010-01-01T05:16:00.000Z,Drug Stores and Pharmacies,0,2010,1,6,Friday,5,Night


In [0]:
silver_tables = ["cards", "users", "transactions"]

print("=" * 70)
print("SILVER TRANSFORMATION COMPLETE")
print("=" * 70)

for table_name in silver_tables:
    full_table_name = f"{SILVER_DB}.{table_name}"
    print(f"Counting {full_table_name} ...")
    row_count = spark.table(full_table_name).count()
    col_count = len(spark.table(full_table_name).columns)
    print(f"{full_table_name}: {row_count:,} rows, {col_count} columns")

print("=" * 70)

print("Fraud distribution:")
display(
    spark.table(f"{SILVER_DB}.transactions")
    .groupBy("is_fraud")
    .count()
    .orderBy("is_fraud")
)

print("MCC join quality:")
display(
    spark.table(f"{SILVER_DB}.transactions")
    .selectExpr(
        "count(*) as total_transactions",
        "sum(case when mcc_description is null then 1 else 0 end) as missing_mcc_description",
        "sum(case when is_fraud = 1 then 1 else 0 end) as fraud_transactions"
    )
)

SILVER TRANSFORMATION COMPLETE
Counting demoworkspacejoby.fraud_silver.cards ...
demoworkspacejoby.fraud_silver.cards: 6,146 rows, 13 columns
Counting demoworkspacejoby.fraud_silver.users ...
demoworkspacejoby.fraud_silver.users: 2,000 rows, 14 columns
Counting demoworkspacejoby.fraud_silver.transactions ...
demoworkspacejoby.fraud_silver.transactions: 220,083 rows, 20 columns
Fraud distribution:


is_fraud,count
0,219648
1,435


MCC join quality:


total_transactions,missing_mcc_description,fraud_transactions
220083,0,435
